
# Support Vector Machine (SVM) — Sổ tay thực hành 🇻🇳

**Mục tiêu file:** Trả lời các câu hỏi 1→7 về SVM và kèm mã Python chạy được với scikit-learn.

**Gợi ý:** Sau khi mở, vào **Kernel → Restart & Run All** để chạy toàn bộ.



## 1) SVM hoạt động thế nào? (Hyperplane & Margin)

**Siêu phẳng (hyperplane)** phân tách hai lớp trong không gian đặc trưng được mô tả bởi phương trình:
$$
\mathbf{w}^\top \mathbf{x} + b = 0
$$

**Lề (margin)** là khoảng cách từ các điểm gần nhất của mỗi lớp đến siêu phẳng. SVM tìm nghiệm sao cho **lề lớn nhất**. Bài toán chuẩn tắc (tuyến tính, tách được) là:
$$
\min_{\mathbf{w}, b} \; \frac{1}{2}\lVert \mathbf{w} \rVert^2 \quad \text{s.t. } y_i(\mathbf{w}^\top \mathbf{x}_i + b) \ge 1 \;\; \forall i
$$

Trực giác: trong vô hạn siêu phẳng có thể phân tách, SVM chọn siêu phẳng **an toàn nhất** (lề lớn).



## 2) Vai trò của **Support Vectors**

- **Support vectors** là các điểm **nằm sát lề** (các ràng buộc hoạt tính).  
- Chúng **quyết định trực tiếp** vị trí của siêu phẳng tối ưu; các điểm ở xa hơn **không ảnh hưởng** đến nghiệm.  
- Nhờ đó mô hình **gọn** và **ổn định** hơn (dựa vào một tập con dữ liệu quan trọng).



## 3) Lề cứng (Hard Margin) vs Lề mềm (Soft Margin)

- **Hard margin:** yêu cầu phân tách **hoàn hảo** (không lỗi). Phù hợp khi dữ liệu thật sự **tách tuyến tính** và **ít nhiễu**.  
- **Soft margin:** cho phép một số **vi phạm lề**; tối ưu hóa cân bằng giữa **lề rộng** và **ít lỗi** nhờ tham số phạt $C$:
$$
\min_{\mathbf{w}, b, \boldsymbol{\xi}} \; \frac{1}{2}\lVert \mathbf{w} \rVert^2 + C \sum_i \xi_i \quad
\text{s.t. } y_i(\mathbf{w}^\top \mathbf{x}_i + b) \ge 1 - \xi_i, \;\; \xi_i \ge 0
$$

**Khi nào dùng lề mềm?** Hầu hết bài toán thực tế (dữ liệu nhiễu/không tách tuyến tính).



## 4) Kernel trong SVM là gì? Khi dùng Linear / Polynomial / RBF?

**Kernel trick:** thay vì biến đổi phi tuyến tường minh, ta dùng **hàm nhân (kernel)** để tính tích vô hướng trong không gian đặc trưng ẩn.

- **Linear**: $K(\mathbf{x},\mathbf{z})=\mathbf{x}^\top\mathbf{z}$  
  - Dùng khi dữ liệu **gần tuyến tính**, đặc trưng **cao và thưa** (VD: bag-of-words).
- **Polynomial**: $K(\mathbf{x},\mathbf{z})=(\gamma\,\mathbf{x}^\top\mathbf{z}+r)^{d}$  
  - Mô hình hóa **tương tác đa thức**; cẩn trọng với **bậc cao** vì dễ quá khớp.
- **RBF (Gaussian)**: $K(\mathbf{x},\mathbf{z})=\exp(-\gamma\lVert \mathbf{x}-\mathbf{z}\rVert^2)$  
  - Linh hoạt, thường là **mặc định mạnh** khi **không biết** dạng phi tuyến. Cần tune đồng thời **$C$** và **$\gamma$**.



## 5) Ý nghĩa tham số **C**

- **$C$ lớn** → phạt nặng lỗi → cố gắng **ít lỗi huấn luyện**, **lề hẹp**, nguy cơ **overfit** cao hơn.  
- **$C$ nhỏ** → cho phép nhiều vi phạm hơn → **lề rộng**, có thể **tổng quát hóa** tốt hơn (tăng bias, giảm variance).

Với RBF, còn có **$\gamma$**:  
- $\gamma$ lớn → ranh giới **gồ ghề** (variance cao).  
- $\gamma$ nhỏ → ranh giới **mượt** (bias cao).



## 6) Code mẫu: SVM phân loại (scikit-learn)

Quy trình: tách X/y → chia train/test (stratify) → **scaling** → chọn kernel/siêu tham số → fit & đánh giá.


In [ ]:

# Demo SVM với dữ liệu toy
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1) Tạo dữ liệu mẫu
X, y = make_classification(
    n_samples=1200, n_features=10, n_informative=5, n_redundant=2,
    class_sep=1.5, random_state=42
)
df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
df["target"] = y

# 2) Tách dữ liệu
X = df.drop(columns=["target"])
y = df["target"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 3) Pipeline: scale -> SVC(RBF)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf"))
])

# 4) Tuning cơ bản
param_grid = {
    "clf__C": [0.1, 1, 10],
    "clf__gamma": ["scale", 0.1, 0.01]
}
grid = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1, scoring="f1_macro")
grid.fit(Xtr, ytr)

print("Best params:", grid.best_params_)
yp = grid.predict(Xte)
print("== Classification Report ==")
print(classification_report(yte, yp))
print("Confusion matrix:\n", confusion_matrix(yte, yp))



## 7) Chuẩn hóa dữ liệu trước SVM: dùng hàm nào? Tại sao quan trọng?

- Dùng **`StandardScaler`** (module `sklearn.preprocessing`) để đưa mỗi đặc trưng về **trung bình 0, độ lệch chuẩn 1**.  
- Với SVM (đặc biệt **RBF/Polynomial**), khoảng cách giữa các điểm bị ảnh hưởng mạnh bởi **thang đo**; không scale có thể làm một số đặc trưng **lấn át** phần còn lại và khiến tối ưu hóa **kém ổn định**.

Ví dụ tích hợp trong `Pipeline` (đã dùng ở phần 6):


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf"))
])
pipe
